In [ ]:
# Librerias
import pandas as pd
import numpy as np
import matplotlib.pyplot as pl
import seaborn as sn

In [ ]:
# Leer Dataset
df = pd.read_csv("../data/raw/dirty_cafe_sales.csv")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Conversión de tipos de datos
# Se convierten columnas numéricas a formato numérico
# fecha a datetime para análisis

cols_num = ["Quantity", "Price Per Unit", "Total Spent"]
for col in cols_num:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["Transaction Date"] = pd.to_datetime(df["Transaction Date"], errors="coerce")

In [ ]:
df.info()

In [ ]:
# Revisar valores nulos
df.isnull().sum()

In [ ]:
missing = df.isnull().mean()*100
missing.sort_values(ascending=False)

In [ ]:
# Exploración de variables categóricas
# Revisación de valores más frecuentes en columnas de tipo texto
# para detectar inconsistencias

cols_text = ["Item", "Payment Method", "Location"]
for col in cols_text:
    print(f"\n-------")
    print(df[col].value_counts(dropna=False).head(10))

In [ ]:
# Reemplazar valores de error y etiquetas de datos faltantes ("ERROR", "UNKNOWN")
# Por NaN para unificar el tratamiento de los datos faltantes
df.replace(["ERROR", "UNKNOWN"], pd.NA, inplace=True)
missing = df.isnull().mean()*100
missing.sort_values(ascending=False)

In [ ]:
cols_fill = ["Location", "Payment Method"]
for col in cols_fill:
    df[col] = df[col].fillna("Unknown")
missing = df.isnull().mean()*100
missing.sort_values(ascending=False)

In [ ]:
# Imputación de valores faltantes en "Price Per Unit" usando la mediana por producto (Item)
# Se ve que cada producto tiene un rango de precios relativamente estable,
# por lo que la mediana por Item se utiliza como estimación robusta frente a outliers

price_map = df.groupby("Item")["Price Per Unit"].median().to_dict()
df["Price Per Unit"] = df["Price Per Unit"].fillna(df["Item"].map(price_map))

In [ ]:
# Imputación de valores faltantes en "Total Spent", "Quantity", "Price Per Unit", cuando es posible calcularlos
mask_total = (
    df["Total Spent"].isna() &
    df["Quantity"].notna() &
    df["Price Per Unit"].notna()
)

df.loc[mask_total, "Total Spent"] = (
    df.loc[mask_total, "Quantity"] * df.loc[mask_total, "Price Per Unit"]
)

mask_quantity = (
    df["Quantity"].isna() &
    df["Total Spent"].notna() &
    df["Price Per Unit"].notna()
)

df.loc[mask_quantity, "Quantity"] = (
    df.loc[mask_quantity, "Total Spent"] / df.loc[mask_quantity, "Price Per Unit"]
)

mask_price = (
    df["Price Per Unit"].isna() &
    df["Quantity"].notna() &
    df["Total Spent"].notna()
)

df.loc[mask_price, "Price Per Unit"] = (
    df.loc[mask_price, "Total Spent"] / df.loc[mask_price, "Quantity"]
)

In [ ]:
missing = df.isnull().mean()*100
missing.sort_values(ascending=False)

In [ ]:
# Se elimina filas con valores faltantes en Item y Transaction Date, porque no es posible analizar ventas sin conocer el producto
df = df.dropna(subset=["Item", "Transaction Date"])

In [ ]:
# Se elimina filas con valores faltantes en Quantity y Total Spent
df = df.dropna(subset=["Quantity", "Total Spent"])
missing = df.isnull().mean()*100
missing.sort_values(ascending=False)

In [ ]:
# Nueva columna fecha separada por mes y día de la semana
df["Month"] = df["Transaction Date"].dt.month_name()
df["Day"] = df["Transaction Date"].dt.day_name()

In [ ]:
df.to_csv("../data/processed/coffee_sales_clean.csv", index=False) 

In [ ]:
df.info()